In [98]:
## read the dataset
dataset = open('input.txt', 'r').read()

In [110]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from collections import Counter

In [100]:
data = sorted(list(set(dataset)))

In [123]:
vocab_size = 300
initialVocab = len(data)
context_length = 6
batch_size = 4
n_embed = 10

In [102]:
## mapping (char to int) and (int, char)
stoi = {s:i for i,s in enumerate(data)}
itos = {i:s for i,s in enumerate(data)}
encoder = lambda l : [stoi[ch] for ch in l]
decoder = lambda d : [itos[id] for id in d]

In [103]:
## 20% of training data set
n3 = int(0.2 * len(dataset))
text = dataset[:n3]
## creating tokens
tokens = encoder(text)
extravocabs = vocab_size - initialVocab

In [104]:
def get_pair(tokens):

    def create_pairs(tokens):
        counter = Counter()
        for pair in zip(tokens[:], tokens[1:]):
            counter[pair]+=1
        return counter
    counter = create_pairs(tokens)
    max_pair = max(counter, key=counter.get)
    return max_pair

In [105]:
## creating vocabulary using BPE
for j in range(extravocabs):
    pair = get_pair(tokens)
    currentTokenId = j + initialVocab
    i = 0
    new_tokens = []
    while i < len(tokens):
        if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == pair:
            st = itos[tokens[i]]+itos[tokens[i+1]]
            stoi[st] = currentTokenId
            itos[currentTokenId] = st
            new_tokens.append(currentTokenId)
            i+=2
        else:
            new_tokens.append(tokens[i])
            i+=1
    tokens = new_tokens

In [ ]:
## Above is the 300 size vocabulary has been created

In [113]:
## tokenize the whole dataset and split in training and validation
tokenization = torch.tensor(encoder(dataset), dtype=torch.long)
n1 = int(0.9 * len(tokenization))
train_dataset = tokenization[:n1]
val_dataset = tokenization[n1:]

In [ ]:
train_dataset

torch.Size([1003854])

In [117]:
## Next Input and output data split with the batch
torch.manual_seed(1337)
def get_batch(split):
    splitToProcess = train_dataset if split == 'train' else val_dataset
    startingPointers = torch.randint(0, len(splitToProcess)-context_length , (batch_size,))
    x = torch.stack([splitToProcess[i:i+context_length] for i in startingPointers])
    y = torch.stack([splitToProcess[i+1:i+context_length+1] for i in startingPointers])
    return x,y
    

In [148]:
x, y = get_batch('train')

In [183]:
class BiagramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.Embedding = nn.Embedding(initialVocab, initialVocab)
    def forward(self, x, out=None):
        logits = self.Embedding(x)
        if out == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            out = out.view(B*T)
            loss = F.cross_entropy(logits,out)
        return logits , loss
    def generate(self, inputToken, max_n_tokens):
        for _ in range(max_n_tokens):
            logits , loss = self(inputToken)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            pred = torch.multinomial(probs, num_samples = 1)
            inputToken = torch.cat((inputToken, pred), dim=1)
        return inputToken


In [184]:
C = BiagramModel()
out , loss = C.forward(x,y)

In [224]:
print("".join(decoder(C.generate(torch.zeros((1,1), dtype=torch.long), 200)[0].tolist())))



A: it feghefeayem le may thinsh thyo:
STI's necerdo ck solll, s semetheley
IUThicorularck
SISomonthinofothye! n thoshe t De?
ICoveanchelefo we his tlifiou t, the, s plsoourdsinde bbe an s in
NCHeset 


In [193]:
## Optimizer
optimizer = torch.optim.AdamW(C.parameters(), lr=1e-3)

In [221]:
batch_size = 32
for _ in range(10000):
    xb, yb = get_batch('train')

    logits, loss = C(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()
    optimizer.step()
print(loss.item())

2.394339084625244
